# YemenVision-MTMC — المرحلة ٨ — النسخة الاحترافية: تعلّم المجهول + المشاة والتتبع + مخزن مسارات دائم (دفتر واحد متكامل)

يجمع في دفتر واحد: **(أ)** اكتشاف المركبات/الطرازات التي لا يعرفها النموذج وتعلّمها تراكمياً دون نسيان، **(ب)** كشف المشاة وتتبعهم وتخزين مساراتهم، **(ج)** حفظ كل النماذج المفتوحة المصدر والمُطوَّرة في Drive.

## كيف يعمل (وما معنى «يتعلم صوراً لا يعرفها»)
| الخطوة | ماذا يحدث | من يقرّر؟ |
|---|---|---|
| 5 | يمرّ على صور جديدة، يلتقط المركبات التي **ثقته فيها منخفضة أو «غير معروفة»**، ويجمّع المتشابهة بصرياً في عناقيد | النموذج |
| تسمية | تفتح كل عنقود وتسمّيه (`Hyundai__County__2008-2016`) | **أنت** |
| 6 | يضيف الفئات الجديدة إلى البنك، يقيس على اختبار **ثابت لا يتغير بين الجولات**، ويرفض النسخة إن **نسي** ما تعلمه سابقاً | بوابة آلية |
لا تسميات آلية يُدرَّب عليها: تسميات نموذج ضعيف تكرّر أخطاءه. التعلّم هنا يعتمد على تسميتك أنت. (ويمكنك أيضاً إضافة طراز بالنص فقط في `vehicle_vocab_yemen.csv` دون صور.)

## المشاة: ما يُبنى وما لا يُبنى — بقصد
- **يُبنى:** كشف (YOLO11 COCO) + تتبع (BoT-SORT) + أرقام مجهولة (`P-000001`) تُستنتج من المظهر فقط لربط نفس المارّ عبر المسارات والكاميرات + مسارات وسرعات (بالأمتار إن زوّدت homography) + قاعدة SQLite في Drive.
- **لا يُبنى:** التعرف على الوجوه، الربط بأسماء أو هويات حقيقية، استنتاج الجنس/العمر/العِرق/الدين، أو تخزين صور الأشخاص. هذه حدود مقصودة تقلّل الضرر لو تسرّبت البيانات أو أُسيء استخدامها، وهي ما أنصح به مهنياً.
- **حماية في الكود:** سياسة احتفاظ تحذف المسارات الأقدم (`retention_days`)، وحذف هوية واحدة بطلب (`delete_entity`)، ومطابقة **محافِظة** (دمج شخصين خطأً أسوأ من تفتيت شخص).
- **قانونياً:** تأكد من التفويض واللافتات في مواقع التصوير. وبعض مجموعات ReID الشهيرة جُمعت دون موافقة المصوَّرين (وسُحبت إحداها من مصدرها)، لذا Market-1501 **لا تُستخدم إلا بقبولك الصريح** (`I_ACCEPT_DATASET_TERMS`).

## حدود صادقة
1. **«النموذج العملاق» للمشاة:** لا أملك نموذج ReID عملاقاً مُتحقَّقاً منه ورخصته واضحة وأوزانه قابلة للتنزيل. أستخدم بصمة **OpenCLIP العملاق** نفسه (ZS) مع **مهايئ خطي** تُدرّبه أنت على بياناتك المرخّصة. **جودة مطابقة المشاة عبر الكاميرات غير معروفة حتى تقيسها** (الخلية 10). بدون بياناتك العتبة محافِظة وغير معايرة.
2. **تتبع MOT الحقيقي** (IDF1/MOTA) يحتاج فيديو بحقيقة أرضية؛ الدفتر لا يقدّمه، ويعطي إحصاءات المسارات فقط.
3. **«أداء أعلى»:** الدفتر يختار تلقائياً هل يفعّل TTA (قلب أفقي) بالقياس على التحقق، ويقيس بوابات النسيان. لا أعد برقم.
4. **الرخص:** Ultralytics YOLO تحت AGPL-3.0 (الاستخدام التجاري المغلق يتطلب ترخيصاً)، وأوزان OpenCLIP لها رخص مستقلة؛ سجل `model_zoo/zoo_manifest.json` يذكّرك بذلك.

## التشغيل
T4 مطلوب. شغّل المرحلة ٧ قبل هذا الدفتر (يحتاج `vehicle_giant_bank.pt`). الخلايا بالترتيب؛ ما لا بيانات له يُتخطى ويُعلن `NOT_EVALUATED`. كل شيء يُحفظ في Drive: `model_zoo/` (أوزان مفتوحة)، `models_v8/` (نسخ مرقّمة)، `production/` (الإنتاج)، `tracking/` (القاعدة والمسارات).

In [ ]:
# 0. فحص البيئة — تثبيت الناقص فقط، وعدم العبث بإصدار PyTorch/CUDA الموجود
import sys, subprocess, importlib.util, platform
REQUIRED = {"numpy":"numpy<2.3", "pandas":"pandas", "PIL":"pillow", "sklearn":"scikit-learn", "scipy":"scipy", "cv2":"opencv-python-headless", "open_clip":"open_clip_torch", "ultralytics":"ultralytics>=8.3.0"}
missing=[pkg for mod,pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
# لا يكفي التحقق من وجود NumPy: النسخة السابقة كانت 2.4.6 رغم أن الدفتر يطلب <2.3.
try:
    from importlib.metadata import version as _pkg_version
    from packaging.version import Version as _Version
    if importlib.util.find_spec('numpy') is not None and _Version(_pkg_version('numpy')) >= _Version('2.3'):
        missing.append('numpy<2.3')
except Exception as _version_error:
    print('تعذر فحص إصدار NumPy:', _version_error)
if missing:
    print("تثبيت/تصحيح الحزم:", missing)
    subprocess.check_call([sys.executable,"-m","pip","install","-q",*missing])
import numpy as np, pandas as pd, torch
print("Python:",platform.python_version(),"| NumPy:",np.__version__,"| Pandas:",pd.__version__,"| PyTorch:",torch.__version__)
print("CUDA:",torch.cuda.is_available())
if torch.cuda.is_available(): print("GPU:",torch.cuda.get_device_name(0),"| CUDA runtime:",torch.version.cuda)
else: print("تنبيه: نماذج الرؤية الكبيرة بطيئة على CPU؛ خلايا التشغيل الثقيل ستتخطى بأمان.")


In [ ]:
# ١. الإعداد — كل النماذج المفتوحة المصدر تُحفظ في Drive (HF/torch/ultralytics تخزّن أوزانها هناك)، فلا تُعاد تنزيلاتها كل جلسة
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, re, math, time, random, shutil, hashlib, subprocess, importlib.util, warnings, copy
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
def status(kind, message): print({'ok': '✅', 'pause': '⏸️', 'error': '❌', 'warn': '⚠️'}.get(kind, 'ℹ️'), message)
try:
    from google.colab import drive
    drive.mount('/content/drive'); IN_COLAB = True
    DRIVE_ROOT = Path('/content/drive/MyDrive/YemenVision_MTMC'); LOCAL = Path('/content/stage8_work')
except Exception:
    IN_COLAB = False; DRIVE_ROOT = Path(os.environ.get('YV_DRIVE_ROOT', './YemenVision_MTMC')); LOCAL = Path(os.environ.get('YV_LOCAL', './stage8_work'))
    status('warn', 'لا يعمل داخل Colab؛ سيُستخدم مجلد محلي')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN_DIR = DRIVE_ROOT / 'stage8' / RUN_ID; PROD_DIR = DRIVE_ROOT / 'production'; ZOO = DRIVE_ROOT / 'model_zoo'
MODELS_V8 = DRIVE_ROOT / 'models_v8'; TRACK_DIR = DRIVE_ROOT / 'tracking'
for d in (DRIVE_ROOT, LOCAL, RUN_DIR, ZOO / 'hf', ZOO / 'torch', ZOO / 'ultralytics', MODELS_V8, TRACK_DIR): d.mkdir(parents=True, exist_ok=True)
# يجب ضبطها قبل استيراد المكتبات
os.environ['HF_HOME'] = str(ZOO / 'hf'); os.environ['TORCH_HOME'] = str(ZOO / 'torch'); os.environ['OPEN_CLIP_CACHE_DIR'] = str(ZOO / 'hf')
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for c in iter(lambda: f.read(8 << 20), b''): h.update(c)
    return h.hexdigest()
def atomic_copy(src, dst):
    dst = Path(dst); dst.parent.mkdir(parents=True, exist_ok=True); tmp = dst.with_name(dst.name + '.tmp'); shutil.copyfile(src, tmp); os.replace(tmp, dst)
def _jd(o):
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.ndarray): return o.tolist()
    return str(o)
def save_json(path, obj):
    Path(path).parent.mkdir(parents=True, exist_ok=True); Path(path).write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=_jd), encoding='utf-8')
need = {'open_clip': 'open_clip_torch', 'sklearn': 'scikit-learn', 'scipy': 'scipy', 'PIL': 'pillow', 'ultralytics': 'ultralytics>=8.3.0', 'cv2': 'opencv-python-headless'}
missing = [s for m, s in need.items() if importlib.util.find_spec(m) is None]
if missing: print('تثبيت:', missing); subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
import torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
from ultralytics import settings as _ul; _ul.update({'weights_dir': str(ZOO / 'ultralytics')})
HAS_CUDA = torch.cuda.is_available(); DEVICE = torch.device('cuda' if HAS_CUDA else 'cpu')
status('ok' if HAS_CUDA else 'warn', f'torch={torch.__version__} | ' + (f'GPU: {torch.cuda.get_device_name(0)}' if HAS_CUDA else 'لا GPU — فعّل T4'))
status('ok', f'Drive: {DRIVE_ROOT} | مخزن النماذج: {ZOO} | الجولة: {RUN_ID}')


In [ ]:
# ٢. الإعدادات + سياسة الخصوصية (تُطبَّق في الكود، لا تعليقات فقط)
CONFIG8 = {
    'seed': 42,
    # ---- مركبات: اكتشاف «المجهول» وتعلّمه ----
    'new_images_dir': '',            # صور جديدة (إطارات فيديو مثلاً) للبحث فيها عن مركبات/طرازات لا يعرفها النموذج
    'unknown_keep_conf': 0.60,       # مقصوصة ثقتها دون هذا (أو «غير معروف») تدخل حزمة المراجعة
    'cluster_distance': 0.15,        # مسافة cosine لتجميع نفس الطراز معاً (أصغر = تجميع أدق وعناقيد أكثر)
    'min_cluster': 3, 'max_images': 3000, 'max_per_cluster_preview': 12,
    'crops_dir': str(DRIVE_ROOT / 'vehicle_makemodel_crops'),    # فئاتك المسمّاة (تتراكم عبر الجولات)
    'vocab_csv': str(DRIVE_ROOT / 'vehicle_vocab_yemen.csv'),
    'min_images_per_class': 8, 'val_frac': 0.15, 'test_frac': 0.25, 'neardup_hamming': 6, 'unknown_target_precision': 0.90, 'boot': 1000,
    'max_forgetting_drop': 0.02,     # أقصى تراجع مسموح في دقة الفئات القديمة بعد إضافة جديدة
    'min_test_top1_ci_low': 0.50,
    # ---- مشاة ومركبات: كشف وتتبع ----
    'person_detector': 'yolo11m.pt',  # YOLO11m COCO (فئة 0 = شخص). رخصة Ultralytics: AGPL-3.0 (الاستخدام التجاري يتطلب ترخيصاً مؤسسياً)
    'tracker': 'botsort.yaml', 'det_conf': 0.35, 'vid_stride': 1,
    'person_match_cos': 0.93,         # عتبة مطابقة المشاة بين المسارات: محافِظة وغير معايرة حتى تقيس على بياناتك (انظر الخلية 10)
    'vehicle_match_cos': 0.85,
    'vehicle_fallback_model': 'yolo11m.pt',  # YOLO11m COCO للمركبات
    'vehicle_fallback_classes': [2, 3, 5, 7],
    'use_custom_vehicle_detector': False,   # اجعله True فقط إذا كان لديك كاشف مخصص وتعرف أنه مدرّب ومختبر
    'custom_vehicle_detector_path': '',     # مسار best.pt المخصص؛ عند False سيستخدم YOLO11m.pt  # car, motorcycle, bus, truck
    'videos': [],                     # [{'path': '/content/drive/.../cam1.mp4', 'camera': 'cam1', 'start_utc': '2026-11-21T08:00:00'}]
    'homography': None,               # اختياري: {'image': [[x,y]×4], 'world_m': [[X,Y]×4]} لتحويل المسار إلى أمتار
    # ---- خصوصية ----
    'retention_days': 30,             # حذف المسارات الأقدم تلقائياً
    'store_person_images': False,     # لا نخزّن صور المشاة أبداً (بصمات مجهولة + مسارات فقط)
    # ---- تقييم/تكييف المشاة (اختياري، ببياناتك أنت) ----
    'person_labeled_dir': '',         # <معرّف>/<كاميرا>_*.jpg جمعتها بموافقة/تفويض قانوني؛ ≥30 معرّفاً و≥2 كاميرا لكل معرّف
    'market1501_dir': '', 'I_ACCEPT_DATASET_TERMS': False,   # Market-1501 للبحث فقط وجُمعت دون موافقة المصوَّرين؛ لا تُستخدم إلا بقبولك صراحة
}
random.seed(CONFIG8['seed']); np.random.seed(CONFIG8['seed']); torch.manual_seed(CONFIG8['seed'])
CONFIG6 = CONFIG7 = CONFIG8; CONFIG8.setdefault('multiword_makes', ['land rover', 'alfa romeo', 'aston martin', 'mercedes benz', 'rolls royce', 'great wall'])
CONFIG8.setdefault('aliases', {'mercedes': 'mercedes benz', 'benz': 'mercedes benz', 'landcruiser': 'land cruiser'})
save_json(RUN_DIR / 'config.json', CONFIG8)
print("""سياسة الخصوصية المطبّقة في هذا الدفتر:
 • لا تعرّف على الوجوه ولا ربط بأسماء/هويات حقيقية: المشاة يُعطون أرقاماً مجهولة (P-000001) تُستنتج من المظهر فقط.
 • لا استدلال على خصائص حساسة (جنس، عمر، عِرق، دين). لا نخزّن صور الأشخاص.
 • سياسة احتفاظ (retention_days) وحذف كيان واحد بطلب (store.delete_entity).
 • تأكد من التفويض القانوني واللافتات في مواقع التصوير قبل أي نشر ميداني.""")


In [ ]:
# دوال مساعدة من المراحل ٥–٧ (تحليل أسماء الفئات، بصمات dHash وتقسيم بلا تسرب، مقاييس ReID، مفردات)
import difflib

def norm_text(s):
    s = str(s).lower().replace('_', ' ').replace('-', ' ')
    return re.sub(r'\s+', ' ', re.sub(r'[^a-z0-9 ]', ' ', s)).strip()

def parse_user_label(folder_name):
    parts = folder_name.split('__'); make = norm_text(parts[0]); model = norm_text(parts[1]) if len(parts) > 1 else ''
    make = CONFIG6['aliases'].get(make.replace(' ', ''), CONFIG6['aliases'].get(make, make)); model = CONFIG6['aliases'].get(model.replace(' ', ''), model)
    y0 = y1 = None
    if len(parts) > 2:
        ys = re.findall(r'(?:19|20)\d{2}', parts[2]); y0 = int(ys[0]) if ys else None; y1 = int(ys[-1]) if ys else y0
    return {'folder': folder_name, 'make': make, 'model': model, 'y0': y0, 'y1': y1}

POPCOUNT = np.array([bin(i).count('1') for i in range(256)], dtype=np.uint8)

def dhash(path, size=8):
    with Image.open(path) as im:
        g = im.convert('L').resize((size + 1, size), Image.BILINEAR)
    a = np.asarray(g, dtype=np.int16); return np.packbits((a[:, 1:] > a[:, :-1]).flatten())

def hamming_matrix(A, B): return POPCOUNT[np.bitwise_xor(A[:, None, :], B[None, :, :])].sum(-1)

def union_find_groups(n, pairs):
    parent = list(range(n))
    def find(x):
        while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]
        return x
    for i, j in pairs:
        a, b = find(i), find(j)
        if a != b: parent[b] = a
    g = {}
    for i in range(n): g.setdefault(find(i), []).append(i)
    return list(g.values())

def split_class_groups(n_images, groups, test_frac, val_frac, seed):
    # يعيد 'train'/'val'/'test' لكل صورة داخل فئة واحدة، بمجموعات كاملة
    rng = np.random.default_rng(seed); order = rng.permutation(len(groups)); lab = ['train'] * n_images
    if len(groups) < 3: return lab, False
    need_te, need_va = max(1, round(test_frac * n_images)), max(1, round(val_frac * n_images)); te = va = 0
    cap_te, cap_va = max(need_te * 1.5, need_te + 1), max(need_va * 1.5, need_va + 1)   # مجموعة ضخمة لا تبتلع الفئة كلها
    for gi in order:
        n_train_left = sum(1 for l in lab if l == 'train'); g = groups[gi]
        if te < need_te and te + len(g) <= cap_te and n_train_left - len(g) >= n_images * 0.5:
            for i in g: lab[i] = 'test'
            te += len(g)
        elif va < need_va and va + len(g) <= cap_va and n_train_left - len(g) >= n_images * 0.5:
            for i in g: lab[i] = 'val'
            va += len(g)
    return lab, ('test' in lab and 'train' in lab)

def evaluate_reid(sim, q_pid, g_pid, q_cam, g_cam, topk=(1, 5, 10)):
    # بروتوكول VeRi القياسي: تُستبعد من gallery الصور بنفس الهوية ونفس الكاميرا
    ap_list, r_rows, kept = [], [], []
    K = max(topk)
    for i in range(len(sim)):
        order = np.argsort(-sim[i], kind='stable')
        junk = (g_pid[order] == q_pid[i]) & (g_cam[order] == q_cam[i])
        matches = (g_pid[order[~junk]] == q_pid[i]).astype(np.int64)
        if matches.sum() == 0: continue
        hits = np.cumsum(matches)
        ap_list.append(float(((hits / (np.arange(len(matches)) + 1)) * matches).sum() / matches.sum()))
        r0 = np.minimum(hits, 1)[:K]; r_rows.append(np.pad(r0, (0, max(0, K - len(r0))))); kept.append(i)
    r = np.stack(r_rows)
    out = {'mAP': float(np.mean(ap_list)), 'per_query_ap': np.array(ap_list), 'per_query_r1': r[:, 0].astype(float), 'kept': np.array(kept)}
    for k in topk: out[f'rank{k}'] = float(r[:, k - 1].mean())
    return out

def boot_ci(x, n=2000, seed=0):
    rng = np.random.default_rng(seed); x = np.asarray(x, float)
    m = x[rng.integers(0, len(x), (n, len(x)))].mean(1)
    return float(x.mean()), float(np.quantile(m, 0.025)), float(np.quantile(m, 0.975))

def vocab_entries(vocab_df):
    ents = []
    for r in vocab_df.itertuples():
        y0, y1 = (None if pd.isna(r.year_from) else int(r.year_from)), (None if pd.isna(r.year_to) else int(r.year_to))
        base = f'{r.make} {r.model}'.strip(); names = [base]
        if y0: names += [f'{base} {y0}-{y1 or "present"}', f'{(y0 + (y1 or y0 + 8)) // 2} {base}']
        if r.generation: names.append(f'{base} {r.generation}')
        ents.append({'make': str(r.make), 'model': str(r.model), 'generation': str(r.generation), 'y0': y0, 'y1': y1, 'names': names})
    return ents
status('ok', 'الدوال المساعدة جاهزة')


In [ ]:
# ٢ مكرر. فحص وإصلاح ذاكرة OpenCLIP قبل تحميل أي نموذج
# سبب الخطأ السابق: ملف checkpoint بحجم 79 بايت داخل HF cache؛ يجب إصلاحه قبل الخلية التالية.
from pathlib import Path
import os

HF_CACHE = Path(os.environ.get("HF_HOME", str(ZOO / "hf")))

def repair_openclip_cache(cache_root=HF_CACHE, min_bytes=1_000_000, force=False):
    """يحذف ملفات أوزان OpenCLIP المبتورة فقط، مع إزالة blob الهدف عند وجود symlink."""
    removed = []
    if not cache_root.exists():
        cache_root.mkdir(parents=True, exist_ok=True)
        return removed

    for f in cache_root.rglob("*"):
        if not (f.is_file() or f.is_symlink()):
            continue
        # لا نحذف أوزان PyTorch العامة؛ نستهدف ملف OpenCLIP المحدد فقط.
        if f.name != "open_clip_pytorch_model.bin":
            continue
        try:
            size = f.stat().st_size
            if size >= min_bytes and not force:
                continue
            resolved = f.resolve(strict=False)
            candidates = [f]
            if resolved != f:
                candidates.append(resolved)
            for candidate in candidates:
                try:
                    if candidate.exists():
                        candidate.unlink()
                        removed.append(str(candidate))
                except OSError as exc:
                    print(f"تعذر حذف الملف التالف {candidate}: {exc}")
        except OSError as exc:
            print(f"تعذر فحص {f}: {exc}")

    print(f"فحص OpenCLIP cache: أزيل {len(removed)} ملف/رابط تالف.")
    return removed

repair_openclip_cache()


In [ ]:
# ٣. الوحدة المتكاملة (مركبات + مشاة + تتبع + مخزن مسارات + هويات مجهولة) — تُكتب في Drive وتُحمَّل منه
UNIFIED_SRC = '''import json, math, sqlite3, time, os, shutil
from datetime import datetime, timezone
from pathlib import Path
import numpy as np

# ---------------------------------------------------------------- أدوات رقمية بلا torch (قابلة للاختبار)
def aggregate_log(prob_list, weights=None):
    P = np.clip(np.stack([np.asarray(p, np.float64) for p in prob_list]), 1e-9, 1.0)
    w = np.ones(len(P)) if weights is None else np.asarray(weights, np.float64)
    lp = (np.log(P) * (w / w.sum())[:, None]).sum(0); e = np.exp(lp - lp.max()); return e / e.sum()

def l2n(x, axis=-1):
    x = np.asarray(x, np.float64); return x / (np.linalg.norm(x, axis=axis, keepdims=True) + 1e-12)

def decide(b, vE, uP, cP, det_cls=None):
    out = {'vehicle_type': det_cls}; ci = int(np.argmax(cP))
    out['color'] = {'name': b['color']['names'][ci], 'p': float(cP[ci]), 'reliable': bool(cP[ci] >= 0.5)}
    u = b.get('user')
    if uP is not None and u is not None:
        k = int(np.argmax(uP))
        if uP[k] >= u['thr']:
            info = u['info'][k]; out.update(source='trained_on_your_data', make=info['make'], model=info['model'], years=info['years'], confidence=float(uP[k])); return out
    key_idx = np.asarray(b['vocab']['key_index']); nk = int(key_idx.max()) + 1
    vk = np.zeros(nk); np.add.at(vk, key_idx, vE); j = int(vk.argmax())
    if vk[j] >= b['vocab_thr']:
        members = np.where(key_idx == j)[0]; e = b['vocab']['entries'][int(members[np.argmax(vE[members])])]
        out.update(source='zero_shot_giant' + ('' if b.get('vocab_thr_calibrated') else ' (uncalibrated)'), make=e['make'], model=e['model'],
                   generation=e['generation'] or None, years=[e['y0'], e['y1']] if e['y0'] else None, confidence=float(vk[j]))
    else:
        top = np.argsort(-vk)[:3]
        out.update(source='unknown', make=None, model=None, years=None, confidence=float(vk[j]),
                   top3=[{'label': b['vocab']['uniq_keys'][int(t)], 'p': float(vk[t])} for t in top])
    return out

def homography_from_points(img_pts, world_pts):
    # DLT: 4+ نقاط صورة ↔ إحداثيات أرضية بالمتر. يعيد مصفوفة 3×3
    A = []
    for (x, y), (X, Y) in zip(img_pts, world_pts):
        A.append([-x, -y, -1, 0, 0, 0, x * X, y * X, X]); A.append([0, 0, 0, -x, -y, -1, x * Y, y * Y, Y])
    _, _, vt = np.linalg.svd(np.asarray(A, np.float64)); H = vt[-1].reshape(3, 3); return H / H[2, 2]

def apply_homography(H, pts):
    p = np.asarray(pts, np.float64).reshape(-1, 2); q = np.c_[p, np.ones(len(p))] @ np.asarray(H).T; return q[:, :2] / q[:, 2:3]

def path_stats(xy, t):
    # طول المسار (وحدة الإحداثيات)، المدة، متوسط السرعة
    xy, t = np.asarray(xy, np.float64), np.asarray(t, np.float64)
    if len(xy) < 2: return {'length': 0.0, 'duration_s': 0.0, 'mean_speed': 0.0}
    length = float(np.linalg.norm(np.diff(xy, axis=0), axis=1).sum()); dur = float(t[-1] - t[0])
    return {'length': length, 'duration_s': dur, 'mean_speed': length / dur if dur > 0 else 0.0}

# ---------------------------------------------------------------- مخزن المسارات (SQLite) — بلا صور، أرقام مجهولة فقط
class TrackStore:
    def __init__(self, path):
        self.path = str(path); Path(self.path).parent.mkdir(parents=True, exist_ok=True)
        self.db = sqlite3.connect(self.path)
        self.db.executescript("""
        CREATE TABLE IF NOT EXISTS entities(gid TEXT PRIMARY KEY, kind TEXT, created REAL, last_seen REAL, n_tracks INTEGER, emb BLOB, attrs TEXT);
        CREATE TABLE IF NOT EXISTS tracks(uid TEXT PRIMARY KEY, kind TEXT, camera TEXT, local_id INTEGER, gid TEXT, t0 REAL, t1 REAL, n_points INTEGER, stats TEXT, match_sim REAL);
        CREATE TABLE IF NOT EXISTS points(uid TEXT, t REAL, x REAL, y REAL, w REAL, h REAL, conf REAL, gx REAL, gy REAL);
        CREATE INDEX IF NOT EXISTS ix_points_uid ON points(uid); CREATE INDEX IF NOT EXISTS ix_tracks_gid ON tracks(gid);""")
        self.db.commit()
    def close(self): self.db.commit(); self.db.close()
    def next_gid(self, kind):
        prefix = {'person': 'P', 'vehicle': 'V'}.get(kind, 'X')
        row = self.db.execute("SELECT MAX(CAST(SUBSTR(gid, 3) AS INTEGER)) FROM entities WHERE kind=?", (kind,)).fetchone()
        n = int(row[0] or 0)
        return f'{prefix}-{n + 1:06d}'
    def gallery(self, kind, embedding_dim=None):
        rows = self.db.execute('SELECT gid, emb, n_tracks, last_seen FROM entities WHERE kind=?', (kind,)).fetchall()
        decoded = []
        for r in rows:
            e = np.frombuffer(r[1], np.float32)
            if e.size == 0 or (embedding_dim is not None and e.size != int(embedding_dim)):
                continue
            decoded.append((r[0], e, r[2], r[3]))
        if not decoded:
            return [], np.zeros((0, int(embedding_dim or 1))), np.zeros(0, int), np.zeros(0)
        dims = {len(x[1]) for x in decoded}
        if len(dims) != 1:
            return [], np.zeros((0, int(embedding_dim or 1))), np.zeros(0, int), np.zeros(0)
        return ([x[0] for x in decoded],
                np.stack([x[1] for x in decoded]).astype(np.float64),
                np.array([x[2] for x in decoded]),
                np.array([x[3] for x in decoded]))
    def upsert_entity(self, gid, kind, emb, t0, t1, attrs=None):
        emb = l2n(emb).astype(np.float32); row = self.db.execute('SELECT emb, n_tracks, created FROM entities WHERE gid=?', (gid,)).fetchone()
        if row is None:
            self.db.execute('INSERT INTO entities VALUES(?,?,?,?,?,?,?)', (gid, kind, t0, t1, 1, emb.tobytes(), json.dumps(attrs or {})))
        else:
            old, n = np.frombuffer(row[0], np.float32).astype(np.float64), row[1]; new = l2n((old * n + emb.astype(np.float64)) / (n + 1)).astype(np.float32)
            self.db.execute('UPDATE entities SET emb=?, n_tracks=n_tracks+1, last_seen=MAX(last_seen,?) WHERE gid=?', (new.tobytes(), t1, gid))
    def overlaps(self, gid, camera, t0, t1):
        return self.db.execute('SELECT COUNT(*) FROM tracks WHERE gid=? AND camera=? AND t0<=? AND t1>=?', (gid, camera, t1, t0)).fetchone()[0] > 0
    def last_in_camera(self, gid, camera):
        r = self.db.execute('SELECT MAX(t1) FROM tracks WHERE gid=? AND camera=?', (gid, camera)).fetchone()[0]; return r
    def add_track(self, uid, kind, camera, local_id, gid, pts, sim, stats):
        # pts: مصفوفة (n, 8): t,x,y,w,h,conf,gx,gy
        pts = np.asarray(pts, np.float64)
        self.db.execute('DELETE FROM points WHERE uid=?', (uid,))
        self.db.execute('INSERT OR REPLACE INTO tracks VALUES(?,?,?,?,?,?,?,?,?,?)', (uid, kind, camera, int(local_id), gid, float(pts[0, 0]), float(pts[-1, 0]), len(pts), json.dumps(stats), float(sim)))
        self.db.executemany('INSERT INTO points VALUES(?,?,?,?,?,?,?,?,?)', [(uid, *map(float, r)) for r in pts]); self.db.commit()
    def purge_older_than(self, days, now=None):
        # سياسة الاحتفاظ: حذف التتبعات والنقاط والكيانات التي انتهت قبل N يوماً
        cutoff = (now if now is not None else time.time()) - days * 86400
        uids = [r[0] for r in self.db.execute('SELECT uid FROM tracks WHERE t1<?', (cutoff,)).fetchall()]
        for i in range(0, len(uids), 500):
            ch = uids[i:i + 500]; q = ','.join('?' * len(ch))
            self.db.execute(f'DELETE FROM points WHERE uid IN ({q})', ch); self.db.execute(f'DELETE FROM tracks WHERE uid IN ({q})', ch)
        self.db.execute('DELETE FROM entities WHERE last_seen<?', (cutoff,)); self.db.commit(); return len(uids)
    def delete_entity(self, gid):
        # حق المحو: حذف كيان واحد وكل مساراته
        uids = [r[0] for r in self.db.execute('SELECT uid FROM tracks WHERE gid=?', (gid,)).fetchall()]
        for u in uids: self.db.execute('DELETE FROM points WHERE uid=?', (u,))
        self.db.execute('DELETE FROM tracks WHERE gid=?', (gid,)); self.db.execute('DELETE FROM entities WHERE gid=?', (gid,)); self.db.commit(); return len(uids)
    def summary(self):
        q = lambda s: self.db.execute(s).fetchall()
        return {'entities': dict(q('SELECT kind, COUNT(*) FROM entities GROUP BY kind')),
                'tracks': dict(q('SELECT kind, COUNT(*) FROM tracks GROUP BY kind')),
                'points': q('SELECT COUNT(*) FROM points')[0][0]}
    def trajectory(self, gid):
        return self.db.execute('SELECT t.camera, p.t, p.x, p.y, p.gx, p.gy FROM points p JOIN tracks t ON p.uid=t.uid WHERE t.gid=? ORDER BY p.t', (gid,)).fetchall()
    def export_csv(self, out_csv, kind=None):
        import csv
        rows = self.db.execute('SELECT t.gid, t.camera, t.uid, p.t, p.x, p.y, p.w, p.h, p.gx, p.gy FROM points p JOIN tracks t ON p.uid=t.uid' + (' WHERE t.kind=?' if kind else '') + ' ORDER BY t.gid, p.t', (kind,) if kind else ()).fetchall()
        with open(out_csv, 'w', newline='', encoding='utf-8') as f:
            w = csv.writer(f); w.writerow(['global_id', 'camera', 'track_uid', 't_epoch', 'x', 'y', 'w', 'h', 'ground_x', 'ground_y']); w.writerows(rows)
        return len(rows)

# ---------------------------------------------------------------- مطابقة الهوية المجهولة بين المسارات
class IdentityMatcher:
    # قاعدة محافِظة: دمج خاطئ لشخصين أسوأ من تفتيت شخص واحد، لذا نشترط عتبة تشابه + هامشاً عن ثاني أفضل مرشح + عدم التزامن في نفس الكاميرا
    def __init__(self, store, kind, thr, margin=0.02, max_gap_same_cam=120.0, max_gap_cross_cam=7200.0):
        self.s, self.kind, self.thr, self.margin, self.gs, self.gc = store, kind, thr, margin, max_gap_same_cam, max_gap_cross_cam
    def assign(self, emb, camera, t0, t1, attrs=None):
        emb = l2n(emb)
        gids, G, _, last = self.s.gallery(self.kind, embedding_dim=emb.shape[0])
        best_gid, best_sim = None, -1.0
        if len(gids):
            if G.ndim != 2 or G.shape[1] != emb.shape[0]:
                G = np.zeros((0, emb.shape[0]), dtype=np.float64); gids = []
            sims = G @ emb if len(gids) else np.zeros(0); order = np.argsort(-sims); cands = []
            for j in order:
                if sims[j] < self.thr: break
                g = gids[j]
                if self.s.overlaps(g, camera, t0, t1): continue
                lc = self.s.last_in_camera(g, camera)
                gap = (t0 - lc) if lc is not None else (t0 - last[j])
                if gap < 0 or gap > (self.gs if lc is not None else self.gc): continue
                cands.append((g, float(sims[j])))
            if cands:
                second = cands[1][1] if len(cands) > 1 else -1.0
                if cands[0][1] - second >= self.margin: best_gid, best_sim = cands[0]
        if best_gid is None:
            gid = self.s.next_gid(self.kind); self.s.upsert_entity(gid, self.kind, emb, t0, t1, attrs); return gid, best_sim, True
        self.s.upsert_entity(best_gid, self.kind, emb, t0, t1, attrs); return best_gid, best_sim, False

def letterbox_square(pil, fill=(114, 114, 114)):
    from PIL import Image
    s = max(pil.size); canvas = Image.new('RGB', (s, s), fill); canvas.paste(pil, ((s - pil.width) // 2, (s - pil.height) // 2)); return canvas

# ---------------------------------------------------------------- المشغّل المتكامل (يحتاج torch / open_clip / ultralytics)
class Unified:
    def __init__(self, bank_path, vehicle_det=None, person_det='yolo11l.pt', reid_path=None, drive_root=None, store_path='tracking.db',
                 person_adapter=None, cfg=None, device='cuda'):
        import torch, open_clip
        self.torch, self.dev = torch, device
        self.cfg = {'tracker': 'botsort.yaml', 'conf': 0.35, 'stride': 1, 'lost_frames': 30, 'min_points': 8, 'min_side_px': 48, 'crops_per_track': 8, 'crop_every': 5,
                    'person_match_cos': 0.93, 'vehicle_match_cos': 0.85, 'match_margin': 0.02, 'person_thr_calibrated': False, 'tta_flip': False, 'retention_days': 30,
                    'homography': None}
        self.cfg.update(cfg or {})
        b = torch.load(bank_path, map_location='cpu', weights_only=False); self.b, self.scale = b, b['scale']; self.cfg['tta_flip'] = bool(b.get('tta', self.cfg['tta_flip']))
        try:
            self.clip, _, self.pre = open_clip.create_model_and_transforms(
                b['arch'], pretrained=b['tag'], device=device,
                precision='fp16' if str(device).startswith('cuda') else 'fp32',
                weights_only=False
            )
        except Exception as first_error:
            # محاولة استرداد واحدة عند تلف ملف أوزان OpenCLIP في HF cache.
            if 'repair_openclip_cache' not in globals():
                raise
            print(f'تعذر تحميل OpenCLIP أول مرة: {type(first_error).__name__}: {first_error}')
            repair_openclip_cache(force=True)
            self.clip, _, self.pre = open_clip.create_model_and_transforms(
                b['arch'], pretrained=b['tag'], device=device,
                precision='fp16' if str(device).startswith('cuda') else 'fp32',
                weights_only=False
            )
        self.clip.eval()
        self.V, self.C = b['vocab']['bank'].float().numpy(), b['color']['bank'].float().numpy(); u = b.get('user')
        if u is not None: self.U, self.W, self.bias, self.cls_idx = u['bank'].float().numpy(), u['W'].numpy(), u['b'].numpy(), np.asarray(u['classes_idx'])
        from ultralytics import YOLO
        fallback_name = self.cfg.get('vehicle_fallback_model', 'yolo11m.pt')
        self.vehicle_fallback_classes = list(self.cfg.get('vehicle_fallback_classes', [2, 3, 5, 7]))
        self.vehicle_is_fallback = vehicle_det is None
        try:
            self.vdet = YOLO(str(vehicle_det)) if vehicle_det else YOLO(fallback_name)
        except Exception as det_error:
            print(f'تعذر تحميل كاشف المركبات المخصص {vehicle_det}: {det_error}')
            print(f'سيتم استخدام الكاشف الاحتياطي {fallback_name}')
            self.vdet = YOLO(fallback_name)
            self.vehicle_is_fallback = True
        self.pdet = None
        if person_det:
            try:
                self.pdet = YOLO(str(person_det))
            except Exception as person_error:
                print(f'تعذر تحميل كاشف المشاة {person_det}: {person_error} — سيستمر النظام دون مسار المشاة.')
        self.reid = None
        if reid_path:
            try:
                import sys
                for _loader_dir in (Path(drive_root) / 'production', Path(drive_root)):
                    if _loader_dir.is_dir() and str(_loader_dir) not in sys.path:
                        sys.path.insert(0, str(_loader_dir))
                from yemenvision_loaders import load_reid
                self.reid, self.reid_size, _ = load_reid(reid_path, device)
            except Exception as reid_error:
                print(f'تعذر تحميل ReID ({reid_path})؛ سيستخدم النظام OpenCLIP بدلاً منه: {reid_error}')
                self.reid = None
        self.P_adapter = None
        if person_adapter and Path(person_adapter).is_file():
            try:
                self.P_adapter = torch.load(person_adapter, map_location='cpu', weights_only=False)['W'].numpy()
            except Exception as adapter_error:
                print(f'تعذر تحميل مهايئ المشاة؛ سيتم تجاهله: {adapter_error}')
                self.P_adapter = None
        self.store = TrackStore(store_path)
        self.pm = IdentityMatcher(self.store, 'person', self.cfg['person_match_cos'], self.cfg['match_margin'])
        self.vm = IdentityMatcher(self.store, 'vehicle', self.cfg['vehicle_match_cos'], self.cfg['match_margin'])
    # ---- ميزات
    def feats(self, crops):
        torch = self.torch
        with torch.no_grad():
            x = torch.stack([self.pre(c.convert('RGB')) for c in crops]).to(self.dev); x = x.half() if self.dev == 'cuda' else x
            f = self.clip.encode_image(x).float()
            if self.cfg['tta_flip']: f = f + self.clip.encode_image(torch.flip(x, dims=[3])).float()
        return torch.nn.functional.normalize(f, dim=-1).cpu().numpy()
    @staticmethod
    def _sm(z): z = z - z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)
    def vehicle_probs(self, crops, features=None):
        f = self.feats(crops) if features is None else np.asarray(features)
        vE = self._sm(self.scale * f @ self.V.T); cP = self._sm(self.scale * f @ self.C.T); uP = None; u = self.b.get('user')
        if u is not None:
            K = len(u['classes']); ZS = self._sm(self.scale * f @ self.U.T); pr = np.zeros((len(f), K)); pr[:, self.cls_idx] = self._sm(f @ self.W.T + self.bias)
            uP = self._sm(u['alpha'] * np.log(ZS + 1e-9) + (1 - u['alpha']) * np.log(pr + 1e-9))
        return vE, uP, cP
    def describe_vehicles(self, crops, det_classes=None, as_track=False):
        vE, uP, cP = self.vehicle_probs(crops); det_classes = det_classes or [None] * len(crops)
        if as_track:
            vE, cP = aggregate_log(list(vE))[None], aggregate_log(list(cP))[None]; uP = aggregate_log(list(uP))[None] if uP is not None else None
            det_classes = [max(set(det_classes), key=det_classes.count)]
        return [decide(self.b, vE[i], None if uP is None else uP[i], cP[i], det_classes[i]) for i in range(len(vE))]
    def person_embedding(self, crops, heights=None):
        # استخدام نموذج ReID المدرّب عند توفيره، وإلا OpenCLIP كخط أساس.
        f = self.reid_embed(crops) if self.reid is not None else self.feats([letterbox_square(c) for c in crops])
        w = np.ones(len(f), dtype=np.float64) if heights is None else np.asarray(heights, np.float64)
        e = l2n((f * w[:, None]).sum(0) / max(w.sum(), 1e-12))
        if self.P_adapter is not None:
            if self.P_adapter.shape[0] == e.shape[0]: e = l2n(e @ self.P_adapter)
            else: print('تحذير: أبعاد مهايئ المشاة لا تطابق النموذج الحالي؛ تم تجاهله.')
        return e
    def reid_embed(self, crops):
        import torchvision.transforms as T
        torch = self.torch; tf = T.Compose([T.Resize(self.reid_size), T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
        x = torch.stack([tf(c.convert('RGB')) for c in crops]).to(self.dev)
        with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16, enabled=self.dev == 'cuda'):
            f = self.reid(x).float() + self.reid(torch.flip(x, dims=[3])).float()
        return torch.nn.functional.normalize(f, dim=1).cpu().numpy()
    # ---- تتبّع فيديو
    def process_video(self, video, camera, start_epoch=None, persons=True, vehicles=True, progress_every=500):
        import cv2
        from PIL import Image
        cap = cv2.VideoCapture(str(video)); fps = cap.get(cv2.CAP_PROP_FPS) or 25.0; cap.release(); start = start_epoch if start_epoch is not None else time.time()
        out = {'video': str(video), 'camera': camera, 'persons': [], 'vehicles': []}
        for kind, det, classes in (('person', self.pdet, [0]), ('vehicle', self.vdet, None)):
            if det is None or (kind == 'person' and not persons) or (kind == 'vehicle' and not vehicles): continue
            bufs = {}; H = self.cfg['homography']; fidx = -1
            def finalize(lid, buf):
                if len(buf['pts']) < self.cfg['min_points'] or not buf['crops']: return
                pts = np.asarray(buf['pts'], np.float64); uid = f'{camera}:{kind}:{Path(video).stem}:{lid}:{int(time.time() * 1000)}'
                if H is not None: g = apply_homography(H, pts[:, 1:3]); pts = np.c_[pts[:, :6], g]
                else: pts = np.c_[pts[:, :6], np.full((len(pts), 2), np.nan)]
                stats = path_stats(pts[:, 6:8] if H is not None else pts[:, 1:3], pts[:, 0]); stats['unit'] = 'meter' if H is not None else 'pixel'
                if kind == 'person':
                    emb = self.person_embedding(buf['crops'], buf['hts']); gid, sim, new = self.pm.assign(emb, camera, pts[0, 0], pts[-1, 0]); info = {}
                else:
                    info = self.describe_vehicles(buf['crops'], buf['cls'], as_track=True)[0]
                    emb = self.reid_embed(buf['crops']).mean(0) if self.reid is not None else self.feats(buf['crops']).mean(0)
                    gid, sim, new = self.vm.assign(emb, camera, pts[0, 0], pts[-1, 0], attrs=info)
                self.store.add_track(uid, kind, camera, lid, gid, pts, sim, {**stats, **({'vehicle': info} if info else {})})
                out['persons' if kind == 'person' else 'vehicles'].append({'global_id': gid, 'track': uid, 'new_identity': new, 'match_sim': float(sim), 'stats': stats, **({'description': info} if info else {})})
            kwargs = dict(source=str(video), stream=True, persist=True, tracker=self.cfg['tracker'], conf=self.cfg['conf'], verbose=False, vid_stride=self.cfg['stride'])
            if classes is not None: kwargs['classes'] = classes
            elif kind == 'vehicle' and getattr(self, 'vehicle_is_fallback', False): kwargs['classes'] = self.vehicle_fallback_classes
            for r in det.track(**kwargs):
                fidx += 1; frame_no = fidx * self.cfg['stride']; t = start + frame_no / fps
                if r.boxes is not None and r.boxes.id is not None:
                    img = None
                    for lid, bx, cf, c in zip(r.boxes.id.int().tolist(), r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(), r.boxes.cls.int().tolist()):
                        x1, y1, x2, y2 = [float(v) for v in bx]; buf = bufs.setdefault(lid, {'pts': [], 'crops': [], 'hts': [], 'cls': [], 'last': frame_no, 'n': 0})
                        foot = ((x1 + x2) / 2, y2); buf['pts'].append((t, foot[0], foot[1], x2 - x1, y2 - y1, float(cf))); buf['last'] = frame_no; buf['n'] += 1
                        if buf['n'] % self.cfg['crop_every'] == 1 and len(buf['crops']) < self.cfg['crops_per_track'] and min(x2 - x1, y2 - y1) >= self.cfg['min_side_px']:
                            img = img if img is not None else Image.fromarray(r.orig_img[..., ::-1])
                            buf['crops'].append(img.crop((max(0, x1), max(0, y1), min(img.width, x2), min(img.height, y2)))); buf['hts'].append(y2 - y1); buf['cls'].append(det.names[c])
                for lid in [k for k, v in bufs.items() if frame_no - v['last'] > self.cfg['lost_frames']]: finalize(lid, bufs.pop(lid))
                if progress_every and fidx % progress_every == 0: print(f'  [{kind}] frame {frame_no}')
            for lid, buf in list(bufs.items()): finalize(lid, buf)
        self.store.db.commit(); return out
    def purge(self): return self.store.purge_older_than(self.cfg['retention_days'])
'''
(DRIVE_ROOT / 'yemenvision_unified.py').write_text(UNIFIED_SRC, encoding='utf-8')
exec(UNIFIED_SRC, globals())

def newest(patterns):
    hits = [p for pat in patterns for p in DRIVE_ROOT.glob(pat) if p.is_file()]
    return max(hits, key=lambda p: p.stat().st_mtime) if hits else None
BANK = PROD_DIR / 'vehicle_giant_bank.pt' if (PROD_DIR / 'vehicle_giant_bank.pt').is_file() else newest(['stage7/*/vehicle_giant_bank.pt'])
# لا نختار أوزاناً قديمة تلقائياً؛ قد تكون YOLO11s. الافتراضي الصريح هنا YOLO11m COCO.
VEH_DET = None
if CONFIG8.get('use_custom_vehicle_detector', False):
    _custom_path = Path(CONFIG8.get('custom_vehicle_detector_path', ''))
    if _custom_path.is_file() and _custom_path.stat().st_size > 1024:
        VEH_DET = _custom_path
    else:
        status('warn', f'تم طلب كاشف مخصص لكن المسار غير صالح: {_custom_path} — سيُستخدم YOLO11m COCO')
REID = PROD_DIR / 'reid_best.pt' if (PROD_DIR / 'reid_best.pt').is_file() else None
ADAPTER = PROD_DIR / 'person_adapter.pt' if (PROD_DIR / 'person_adapter.pt').is_file() else None
(PROD_DIR / 'yemenvision_loaders.py').exists() or (DRIVE_ROOT / 'yemenvision_loaders.py').exists() or status('warn', 'yemenvision_loaders.py غير موجود: ReID المركبات سيُتخطى')
status('ok' if BANK else 'error', f'بنك المركبات: {BANK} | كاشف المركبات: {VEH_DET} | ReID: {REID} | مهايئ المشاة: {ADAPTER}')

UNI = None
if BANK is not None and HAS_CUDA:
    db_local = LOCAL / 'tracking.db'
    if (TRACK_DIR / 'tracking.db').is_file() and not db_local.exists(): shutil.copy2(TRACK_DIR / 'tracking.db', db_local)
    cfg = {'tracker': CONFIG8['tracker'], 'conf': CONFIG8['det_conf'], 'stride': CONFIG8['vid_stride'], 'person_match_cos': CONFIG8['person_match_cos'],
           'vehicle_match_cos': CONFIG8['vehicle_match_cos'], 'retention_days': CONFIG8['retention_days'],
           'vehicle_fallback_model': CONFIG8['vehicle_fallback_model'],
           'vehicle_fallback_classes': CONFIG8['vehicle_fallback_classes']}
    if CONFIG8['homography']:
        cfg['homography'] = homography_from_points(CONFIG8['homography']['image'], CONFIG8['homography']['world_m'])
    try:
        if not Path(BANK).is_file() or Path(BANK).stat().st_size < 1024:
            raise FileNotFoundError(f'ملف بنك المركبات غير موجود أو تالف/صغير: {BANK}')
        UNI = Unified(BANK, VEH_DET, CONFIG8['person_detector'], REID, DRIVE_ROOT, db_local, ADAPTER, cfg, 'cuda')
        status('ok', f'جاهز: CLIP={UNI.b["arch"]}/{UNI.b["tag"]} | TTA قلب={UNI.cfg["tta_flip"]} | فئاتك في البنك: {len(UNI.b["user"]["classes"]) if UNI.b.get("user") else 0}')
    except Exception as init_error:
        UNI = None
        status('error', f'تعذر تهيئة وحدة Stage 8؛ تم إيقاف الوحدات التابعة بأمان. السبب: {type(init_error).__name__}: {init_error}')
else:
    status('pause', 'شغّل المرحلة ٧ أولاً (يلزم vehicle_giant_bank.pt) وفعّل GPU')

In [ ]:
# ٤. سجل النماذج المفتوحة المصدر المخزّنة في Drive (المصدر، الحجم، sha256، ملاحظة الرخصة) — تبقى بعد انتهاء الجلسة
LICENSE_NOTES = {'ultralytics': 'YOLO (Ultralytics): AGPL-3.0 — الاستخدام التجاري المغلق يتطلب ترخيص Enterprise',
                 'hf': 'أوزان OpenCLIP/Hugging Face: لكل نموذج رخصته في بطاقته؛ راجعها قبل الاستخدام التجاري',
                 'torch': 'أوزان torchvision/timm: راجع رخصة كل نموذج'}
zoo = []
for sub in ('ultralytics', 'hf', 'torch'):
    for p in sorted((ZOO / sub).rglob('*')):
        if p.is_file() and p.suffix in {'.pt', '.pth', '.safetensors', '.bin', '.onnx'}:
            mb = p.stat().st_size / 2**20
            zoo.append({'group': sub, 'file': str(p.relative_to(ZOO)), 'MB': round(mb, 1), 'sha256': sha256_file(p) if mb < 300 else '(كبير: يُسجَّل الحجم فقط)', 'license_note': LICENSE_NOTES[sub]})
ZOO_DF = pd.DataFrame(zoo)
if len(ZOO_DF): print(ZOO_DF[['group', 'file', 'MB']].to_string(index=False)); save_json(ZOO / 'zoo_manifest.json', zoo)
status('ok', f'{len(zoo)} ملف نموذج مفتوح المصدر محفوظ في {ZOO} (لن يُعاد تنزيله)')
# ملاحظة: النماذج التي دربناها (كاشفك، ReID، بنك المركبات) في production/ و models_v8/ وهي مشتقة من هذه الأوزان المفتوحة


In [ ]:
# ٥. اكتشاف المجهول: مقصوصات لا يعرفها النموذج ← تجميع بالمظهر (نفس الطراز معاً) ← حزمة مراجعة لتسميتها بيدك
from sklearn.cluster import AgglomerativeClustering
def cluster_feats(feats, distance):
    if len(feats) < 2: return np.zeros(len(feats), int)
    return AgglomerativeClustering(n_clusters=None, metric='cosine', linkage='average', distance_threshold=distance).fit_predict(np.asarray(feats))

def mine_unknowns(U_, folder, max_images, keep_conf):
    if U_.vdet is None:
        status('pause', 'لا يوجد كاشف مركبات؛ تم تخطي التنقيب.')
        return [], np.zeros((0, 0)), np.zeros((0, 0)), np.asarray(U_.b['vocab']['key_index']), 0
    paths = [p for p in sorted(Path(folder).rglob('*')) if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}][:max_images]
    crop_dir = RUN_DIR / 'mined' / 'crops'; crop_dir.mkdir(parents=True, exist_ok=True); rows, F_, VE = [], [], []
    key_idx = np.asarray(U_.b['vocab']['key_index'])
    for p in paths:
        im = Image.open(p).convert('RGB'); res = U_.vdet.predict(source=im, conf=CONFIG8['det_conf'], verbose=False)[0]; crops, cls = [], []
        for bx, c in zip(res.boxes.xyxy.cpu().numpy(), res.boxes.cls.cpu().numpy()):
            if getattr(U_, 'vehicle_is_fallback', False) and int(c) not in [2, 3, 5, 7]: continue
            x1, y1, x2, y2 = [float(v) for v in bx]
            if min(x2 - x1, y2 - y1) < 64: continue
            mx, my = 0.04 * (x2 - x1), 0.04 * (y2 - y1); crops.append(im.crop((max(0, x1 - mx), max(0, y1 - my), min(im.width, x2 + mx), min(im.height, y2 + my)))); cls.append(U_.vdet.names[int(c)])
        if not crops: continue
        f = U_.feats(crops); vE, uP, cP = U_.vehicle_probs(crops, features=f)
        for i, cr in enumerate(crops):
            d = decide(U_.b, vE[i], None if uP is None else uP[i], cP[i], cls[i])
            if d['source'] == 'unknown' or d['confidence'] < keep_conf:
                fn = crop_dir / f'{len(rows):06d}.jpg'; cr.save(fn, quality=95); rows.append({'crop': str(fn), 'source_image': str(p), 'vehicle_type': cls[i], 'conf': d['confidence'], 'status': d['source']}); F_.append(f[i]); VE.append(vE[i])
    return rows, np.array(F_), np.array(VE), key_idx, len(paths)

MINED = {}
if UNI is not None and CONFIG8['new_images_dir'] and Path(CONFIG8['new_images_dir']).is_dir():
    rows, F_, VE, key_idx, n_imgs = mine_unknowns(UNI, CONFIG8['new_images_dir'], CONFIG8['max_images'], CONFIG8['unknown_keep_conf'])
    if not rows: status('ok', f'لا مقصوصات مجهولة في {n_imgs} صورة — النموذج واثق من كل ما رآه (أو لم تُكشف مركبات)')
    else:
        lab = cluster_feats(F_, CONFIG8['cluster_distance']); pack = RUN_DIR / 'to_label'; summary = []; uniq = UNI.b['vocab']['uniq_keys']
        for cid in np.unique(lab):
            ix = np.where(lab == cid)[0]
            if len(ix) < CONFIG8['min_cluster']: continue
            cen = F_[ix].mean(0); order = ix[np.argsort(-(F_[ix] @ cen))][:CONFIG8['max_per_cluster_preview']]
            vk = np.zeros(int(key_idx.max()) + 1); np.add.at(vk, key_idx, VE[ix].mean(0)); top = np.argsort(-vk)[:3]
            d_ = pack / f'cluster_{cid:03d}_n{len(ix)}'; d_.mkdir(parents=True, exist_ok=True)
            for j in order: shutil.copy2(rows[j]['crop'], d_ / Path(rows[j]['crop']).name)
            summary.append({'cluster': int(cid), 'n_crops': len(ix), 'types_seen': ','.join(sorted({rows[j]['vehicle_type'] for j in ix})), 'hint1': f'{uniq[top[0]]} ({vk[top[0]]:.2f})',
                            'hint2': f'{uniq[top[1]]} ({vk[top[1]]:.2f})', 'hint3': f'{uniq[top[2]]} ({vk[top[2]]:.2f})', 'folder': d_.name})
        S = pd.DataFrame(summary)
        if not S.empty: S = S.sort_values('n_crops', ascending=False)
        S.to_csv(pack / 'clusters.csv', index=False, encoding='utf-8-sig')
        (pack / 'اقرأني.txt').write_text('لكل مجلد عنقود: افتحه، فإن كانت الصور لطراز واحد فأعد تسمية المجلد إلى الماركة__الطراز__سنةمن-سنةإلى (مثال Hyundai__County__2008-2016) '
                                         'وانقله إلى ' + CONFIG8['crops_dir'] + '. التلميحات (hint) تخمين النموذج لا حقيقة. اختلط العنقود؟ احذف الصور الغريبة.', encoding='utf-8')
        MINED = {'n_images': n_imgs, 'n_unknown_crops': len(rows), 'n_clusters_ge_min': len(S)}; save_json(RUN_DIR / 'mined_summary.json', MINED)
        print(S.to_string(index=False)); status('ok', f'{len(rows)} مقصوصة مجهولة من {n_imgs} صورة → {len(S)} عنقود للتسمية في {pack}')
else:
    status('pause', 'لا new_images_dir: ضع مجلد صور/إطارات جديدة (يُفضَّل من مواقعك) للبحث عن مجهول')


In [ ]:
# ٦. التعلّم التراكمي: يضيف فئاتك الجديدة المسمّاة إلى البنك دون نسيان القديم — تقسيم ثابت، تحسين TTA، حارس نسيان، نسخ مُرقَّمة
from sklearn.linear_model import LogisticRegression
TEMPLATES = ['a photo of a {}.', 'a photo of the {}, a type of vehicle.', 'a front view of a {} on the road.', 'a rear view of a {} on the road.', 'a side view of a {}.', 'a low-resolution photo of a {}.']
import open_clip

def enc_text_bank(U_, names_per_class, templates=TEMPLATES, chunk=128):
    tok = open_clip.get_tokenizer(U_.b['arch']); flat, owner = [], []
    for c, names in enumerate(names_per_class):
        for n in names:
            for t in templates: flat.append(t.format(n)); owner.append(c)
    embs = []
    with torch.no_grad():
        for i in range(0, len(flat), chunk): embs.append(F.normalize(U_.clip.encode_text(tok(flat[i:i + chunk]).to(DEVICE)).float(), dim=-1).cpu())
    E = torch.cat(embs).numpy(); owner = np.array(owner); bank = np.stack([E[owner == c].mean(0) for c in range(len(names_per_class))]); return bank / np.linalg.norm(bank, axis=1, keepdims=True)

def pick_threshold(conf, ok, target):
    order = np.argsort(-conf); prec = np.cumsum(ok[order]) / (np.arange(len(ok)) + 1); good = np.where(prec >= target)[0]
    return (float(conf[order][good.max()]), True) if len(good) else (1.0, False)
def sm_np(z): z = z - z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)

RETRAIN, NEW_BANK = {}, None
CROPS_DIR = Path(CONFIG8['crops_dir']); CROPS_DIR.mkdir(parents=True, exist_ok=True); rows = []
for d in sorted(p for p in CROPS_DIR.iterdir() if p.is_dir()):
    lab = parse_user_label(d.name)
    rows += [{'path': str(p), **lab} for p in sorted(d.rglob('*')) if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}]
if UNI is None or not rows:
    status('pause', f'لا صور مسمّاة في {CROPS_DIR} (أو لا نموذج): سمِّ عناقيد الخلية 8 وانقلها هنا ثم أعد التشغيل')
else:
    DATA = pd.DataFrame(rows); DATA['mmkey'] = DATA.make + ' ' + DATA.model
    # ---- سجل تقسيم ثابت: الصور القديمة تبقى في مكانها (لا تسرّب بين الجولات)، الجديدة فقط تُوزَّع ----
    REG = DRIVE_ROOT / 'vehicle_split_registry.csv'; reg = pd.read_csv(REG) if REG.is_file() else pd.DataFrame(columns=['path', 'split', 'reliable'])
    known = dict(zip(reg.path, reg.split)); new_mask = ~DATA.path.isin(known)
    DATA['split'] = DATA.path.map(known); rel_known = dict(zip(reg.path, reg.reliable)); DATA['reliable'] = DATA.path.map(rel_known)
    if new_mask.any():
        H8 = {p: dhash(p) for p in DATA.path[new_mask]}
        for folder, ix in DATA[new_mask].groupby('folder').groups.items():
            ix = list(ix); hs = np.stack([H8[DATA.path[i]] for i in ix]); dm = hamming_matrix(hs, hs)
            groups = union_find_groups(len(ix), [(a, b) for a, b in np.argwhere(np.triu(dm <= CONFIG8['neardup_hamming'], 1))])
            l_, ok_ = split_class_groups(len(ix), groups, CONFIG8['test_frac'], CONFIG8['val_frac'], CONFIG8['seed'])
            for i, l in zip(ix, l_): DATA.loc[i, 'split'] = l; DATA.loc[i, 'reliable'] = bool(ok_ and len(ix) >= CONFIG8['min_images_per_class'])
    DATA['reliable'] = DATA.reliable.fillna(False).astype(bool)
    # صف «موثوق» يُعاد حسابه للفئة كلها بعد الدمج
    DATA['reliable'] = DATA.groupby('folder').reliable.transform('max') & (DATA.groupby('folder').split.transform(lambda s: (s == 'test').any())) & (DATA.groupby('folder').path.transform('count') >= CONFIG8['min_images_per_class'])
    DATA[['path', 'split', 'reliable']].to_csv(REG, index=False)
    print(DATA.groupby('folder').agg(n=('path', 'count'), train=('split', lambda s: (s == 'train').sum()), val=('split', lambda s: (s == 'val').sum()), test=('split', lambda s: (s == 'test').sum()), reliable=('reliable', 'first')).to_string())

    classes = sorted(DATA.folder.unique()); cid = {c: i for i, c in enumerate(classes)}; y = DATA.folder.map(cid).values; K = len(classes)
    tr, va = (DATA.split == 'train').values, (DATA.split == 'val').values; te = ((DATA.split == 'test') & DATA.reliable).values
    if te.sum() < 20 or K < 2 or va.sum() < 10:
        status('pause', f'بيانات غير كافية للحكم (اختبار={int(te.sum())}, تحقق={int(va.sum())}, فئات={K}) — أضف صوراً')
    else:
        # مفردات (تُعاد بناؤها من CSV لتشمل ما أضفته بالنص فقط)
        VOCAB = pd.read_csv(CONFIG8['vocab_csv']).fillna({'generation': ''}); V_ENT = vocab_entries(VOCAB); V_BANK = enc_text_bank(UNI, [e['names'] for e in V_ENT])
        V_KEYS = [f"{e['make']} {e['model']}".strip().lower() for e in V_ENT]; uniq = sorted(set(V_KEYS)); kidx = [uniq.index(k) for k in V_KEYS]
        u_names = []
        for c in classes:
            r0 = DATA[DATA.folder == c].iloc[0]; nm = [f'{r0.make} {r0.model}'.strip()]
            if pd.notna(r0.y0): nm.append(f'{r0.make} {r0.model} {int(r0.y0)}-{int(r0.y1)}')
            u_names.append(nm)
        UBANK = enc_text_bank(UNI, u_names); cls_mm = np.array([DATA.mmkey[DATA.folder == c].iloc[0] for c in classes])
        # ميزات: بدون وبـ«قلب أفقي» (TTA) — يُختار الأفضل على التحقق
        paths = DATA.path.tolist(); res_by_tta = {}
        for tta in (False, True):
            UNI.cfg['tta_flip'] = tta; FE = np.concatenate([UNI.feats([Image.open(p) for p in paths[i:i + 32]]) for i in range(0, len(paths), 32)])
            ZS = sm_np(UNI.scale * FE @ UBANK.T); best = None
            for C_ in (1, 10, 100):
                clf = LogisticRegression(C=C_, max_iter=2000, class_weight='balanced').fit(FE[tr], y[tr]); accv = float((clf.predict(FE[va]) == y[va]).mean())
                if best is None or accv > best[0]: best = (accv, C_, clf)
            clf = best[2]; PR = np.zeros((len(FE), K)); PR[:, clf.classes_] = clf.predict_proba(FE)
            def fuse(a): return sm_np(a * np.log(ZS + 1e-9) + (1 - a) * np.log(PR + 1e-9))
            alpha = max((float((fuse(a).argmax(1) == y)[va].mean()), a) for a in (0.0, 0.25, 0.5, 0.75, 1.0)); PF = fuse(alpha[1])
            res_by_tta[tta] = {'val_acc': alpha[0], 'alpha': alpha[1], 'PF': PF, 'ZS': ZS, 'PR': PR, 'clf': clf, 'C': best[1], 'FE': FE}
        tta = res_by_tta[True]['val_acc'] > res_by_tta[False]['val_acc'] + 0.005; R = res_by_tta[tta]; UNI.cfg['tta_flip'] = tta
        PF, ZS, PR, clf, FE = R['PF'], R['ZS'], R['PR'], R['clf'], R['FE']
        print(f"TTA قلب أفقي: {'مفعّل' if tta else 'غير مفعّل'} (تحقق: بدون={res_by_tta[False]['val_acc']:.3f} | معه={res_by_tta[True]['val_acc']:.3f})")
        ok_mm = lambda P_: cls_mm[P_.argmax(1)] == cls_mm[y]; ok_f = ok_mm(PF)
        m, lo, hi = boot_ci(ok_f[te], CONFIG8['boot']); m0, lo0, hi0 = boot_ci(ok_mm(ZS)[te], CONFIG8['boot']); dm_, dlo, dhi = boot_ci(ok_f[te].astype(float) - ok_mm(ZS)[te].astype(float), CONFIG8['boot'])
        thr, thr_ok = pick_threshold(PF.max(1)[va], (PF.argmax(1) == y)[va], CONFIG8['unknown_target_precision']); ans = te & (PF.max(1) >= thr)
        # عتبة المفردات
        Vp = sm_np(UNI.scale * FE @ V_BANK.T); Vm = np.zeros((len(FE), len(uniq)))
        for j, k_ in enumerate(kidx): Vm[:, k_] += Vp[:, j]
        vtop = np.array(uniq)[Vm.argmax(1)]; v_ok = vtop == DATA.mmkey.str.lower().values; inv = np.isin(DATA.mmkey.str.lower().values, uniq)
        vthr, vok = pick_threshold(Vm.max(1)[va & inv], v_ok[va & inv], CONFIG8['unknown_target_precision']) if (va & inv).sum() >= 10 else (0.5, False)
        # ---- حارس النسيان: نفس صور الاختبار التي قيست في النسخة السابقة ----
        HIST_P = MODELS_V8 / 'eval_history.json'; hist = json.loads(HIST_P.read_text('utf-8')) if HIST_P.is_file() else {'versions': []}
        version = len(hist['versions']) + 1; forget = None
        if hist['versions']:
            prev = hist['versions'][-1]['per_image_ok']; idx = [i for i in np.where(te)[0] if DATA.path[i] in prev]
            if len(idx) >= 15:
                p_acc = float(np.mean([prev[DATA.path[i]] for i in idx])); n_acc = float(np.mean([ok_f[i] for i in idx])); forget = {'n': len(idx), 'prev_acc': p_acc, 'new_acc': n_acc, 'drop': p_acc - n_acc}
        reasons = []
        if not lo >= CONFIG8['min_test_top1_ci_low']: reasons.append(f'الحد السفلي لدقة الطراز {lo:.3f} < {CONFIG8["min_test_top1_ci_low"]}')
        if dm_ < -0.02: reasons.append('أسوأ من الصفري وحده')
        if not thr_ok: reasons.append('لا عتبة «غير معروف» تبلغ الدقة المستهدفة على التحقق')
        elif ans.sum() and ok_f[ans].mean() < CONFIG8['unknown_target_precision'] - 0.05: reasons.append('دقة الإجابة على الاختبار دون المستهدف')
        if forget and forget['drop'] > CONFIG8['max_forgetting_drop']: reasons.append(f"نسيان: تراجعت دقة الاختبار القديم {forget['drop']:.3f} (>{CONFIG8['max_forgetting_drop']})")
        RETRAIN = {'version': version, 'n_classes': K, 'n_test': int(te.sum()), 'tta_flip': bool(tta), 'alpha_zero_shot_weight': R['alpha'], 'fused_model_acc': m, 'fused_ci': [lo, hi], 'zero_shot_model_acc': m0,
                   'fused_vs_zero_shot': {'delta': dm_, 'ci': [dlo, dhi]}, 'unknown_threshold': thr, 'coverage': float(ans.sum() / te.sum()),
                   'precision_when_answering': float(ok_f[ans].mean()) if ans.sum() else None, 'vocab_thr': vthr, 'vocab_thr_calibrated': bool(vok), 'forgetting': forget, 'pass': not reasons, 'fail_reasons': reasons}
        print(json.dumps(RETRAIN, ensure_ascii=False, indent=1, default=_jd))
        print(pd.DataFrame({'folder': np.array(classes)[y[te]], 'ok': ok_f[te]}).groupby('folder').ok.agg(['mean', 'count']).round(3).to_string())
        info = []
        for c in classes:
            p = parse_user_label(c); info.append({'make': p['make'].title(), 'model': p['model'].title(), 'years': [p['y0'], p['y1']] if p['y0'] else None})
        NEW_BANK = {'arch': UNI.b['arch'], 'tag': UNI.b['tag'], 'scale': UNI.scale, 'run_id': RUN_ID, 'version': version, 'tta': bool(tta),
                    'vocab': {'entries': V_ENT, 'bank': torch.tensor(V_BANK, dtype=torch.float16), 'key_index': kidx, 'uniq_keys': uniq},
                    'color': UNI.b['color'], 'vocab_thr': float(vthr), 'vocab_thr_calibrated': bool(vok),
                    'user': {'classes': classes, 'info': info, 'bank': torch.tensor(UBANK, dtype=torch.float16), 'W': torch.tensor(clf.coef_, dtype=torch.float32), 'b': torch.tensor(clf.intercept_, dtype=torch.float32),
                             'classes_idx': [int(i) for i in clf.classes_], 'alpha': R['alpha'], 'thr': float(thr)}, 'eval': RETRAIN}
        bank_v = MODELS_V8 / f'vehicle_giant_bank_v{version}.pt'; torch.save(NEW_BANK, bank_v)
        hist['versions'].append({'version': version, 'run_id': RUN_ID, 'bank': str(bank_v), 'pass': not reasons, 'classes': classes, 'per_image_ok': {DATA.path[i]: bool(ok_f[i]) for i in np.where(te)[0]}})
        save_json(HIST_P, hist); save_json(RUN_DIR / 'retrain_eval.json', RETRAIN)
        if not reasons:
            dst = PROD_DIR / 'vehicle_giant_bank.pt'
            if dst.is_file(): atomic_copy(dst, DRIVE_ROOT / 'production_archive' / RUN_ID / dst.name)
            atomic_copy(bank_v, dst); status('ok', f'النسخة v{version} اجتازت البوابة وصارت في production/ (القديمة مؤرشفة)')
        else: status('warn', f'النسخة v{version} محفوظة في models_v8 لكنها لم تجتز: ' + ' | '.join(reasons))


In [ ]:
# ٧. المشاة: قياس وتكييف بصمة المظهر على بياناتك المرخّصة (اختياري). بلا هذه البيانات تبقى مطابقة المشاة «غير مُتحقَّق منها».
# لا نتعرّف على أشخاص بأعيانهم: الهدف فقط أن يحمل نفس المارّ رقماً مجهولاً واحداً عبر المسارات.
PERSON = {}
def load_person_df():
    d = Path(CONFIG8['person_labeled_dir']) if CONFIG8['person_labeled_dir'] else None; rows = []
    if d and d.is_dir():
        for idd in sorted(p for p in d.iterdir() if p.is_dir()):
            for p in idd.rglob('*'):
                m = re.match(r'([A-Za-z0-9]+)_', p.name)
                if p.suffix.lower() in {'.jpg', '.jpeg', '.png'} and m: rows.append({'path': str(p), 'pid': idd.name, 'cam': m.group(1)})
        return pd.DataFrame(rows), 'مجلدك المرخّص'
    m = Path(CONFIG8['market1501_dir']) if CONFIG8['market1501_dir'] else None
    if m and m.is_dir():
        if not CONFIG8['I_ACCEPT_DATASET_TERMS']:
            status('pause', 'Market-1501 للبحث فقط وجُمعت صورها دون موافقة المصوَّرين. اضبط I_ACCEPT_DATASET_TERMS=True إن قبلت شروطها ومسؤوليتك.'); return pd.DataFrame(), None
        for split, folder in (('train', 'bounding_box_train'), ('query', 'query'), ('gallery', 'bounding_box_test')):
            for p in sorted((next(iter(m.rglob(folder)), m / folder)).glob('*.jpg')):
                mm = re.match(r'(-?\d+)_c(\d)', p.name)
                if mm and int(mm.group(1)) > 0: rows.append({'path': str(p), 'pid': mm.group(1), 'cam': 'c' + mm.group(2), 'split': split})
        return pd.DataFrame(rows), 'Market-1501 (وافقت على شروطها)'
    return pd.DataFrame(), None

PDF, src = load_person_df()
if UNI is None or PDF.empty:
    status('pause', 'لا بيانات مشاة مُعرَّفة (person_labeled_dir): مطابقة المشاة تعمل بعتبة محافِظة غير معايرة، ولا رقم دقة لها. هذا نقص معلن لا مخفي.')
else:
    ids = sorted(PDF.pid.unique()); cams_per = PDF.groupby('pid').cam.nunique(); ids = [i for i in ids if cams_per[i] >= 2]
    PDF = PDF[PDF.pid.isin(ids)].reset_index(drop=True); rng = np.random.default_rng(CONFIG8['seed']); perm = rng.permutation(ids)
    if len(ids) < 30: status('pause', f'{len(ids)} معرّفاً فقط (≥30 مطلوب)')
    else:
        n = len(ids); tr_i, dv_i, te_i = set(perm[:int(.5 * n)]), set(perm[int(.5 * n):int(.7 * n)]), set(perm[int(.7 * n):])
        FE = np.concatenate([UNI.feats([letterbox_square(Image.open(p)) for p in PDF.path[i:i + 32]]) for i in range(0, len(PDF), 32)])
        def split_q_g(mask):
            # لكل هوية: كاميرا واحدة للاستعلام، وكاميرا مختلفة للمعرض.
            sub = PDF.loc[mask]; q_idx, g_idx = [], []
            for pid, group in sub.groupby('pid'):
                cams = sorted(group.cam.unique())
                if len(cams) < 2: continue
                q_rows = group[group.cam == cams[0]]; g_rows = group[group.cam != cams[0]]
                if len(q_rows) and len(g_rows):
                    q_idx.append(int(q_rows.index[0])); g_idx.extend(int(i) for i in g_rows.index)
            return np.asarray(q_idx,dtype=int), np.asarray(g_idx,dtype=int)
        def eval_emb(E, mask):
            q,g=split_q_g(mask)
            if len(q)==0 or len(g)==0: return None
            return evaluate_reid(E[q] @ E[g].T, PDF.pid.values[q], PDF.pid.values[g], PDF.cam.values[q], PDF.cam.values[g])
        te_m,dv_m,tr_m=PDF.pid.isin(te_i).values,PDF.pid.isin(dv_i).values,PDF.pid.isin(tr_i).values
        base_dv,base_te=eval_emb(FE,dv_m),eval_emb(FE,te_m)
        if base_dv is None or base_te is None:
            status('pause','لا يمكن تكوين Query/Gallery عبر كاميرتين لكل هوية؛ أضف صوراً من كاميرات متعددة.')
        else:
            print(f'[{src}] Baseline validation mAP={base_dv["mAP"]:.4f}, Rank1={base_dv["rank1"]:.4f}; test mAP={base_te["mAP"]:.4f}')
            D_=FE.shape[1]; W=nn.Parameter(torch.eye(D_,device=DEVICE)); opt=torch.optim.AdamW([W],lr=1e-3,weight_decay=1e-3)
            Xt=torch.tensor(FE[tr_m],device=DEVICE,dtype=torch.float32); yt=torch.tensor(pd.factorize(PDF.pid[tr_m])[0],device=DEVICE)
            if len(Xt)>=4 and len(torch.unique(yt))>=2:
                for step in range(250):
                    b_=torch.randperm(len(Xt),device=DEVICE)[:min(256,len(Xt))]; z=F.normalize(Xt[b_]@W,dim=1); yy=yt[b_]
                    d=(2-2*z@z.T).clamp_min(1e-9).sqrt(); same=yy[:,None]==yy[None,:]; eye=torch.eye(len(yy),device=DEVICE,dtype=torch.bool)
                    pos=d.masked_fill(~(same & ~eye),-1).max(1).values; neg=d.masked_fill(same,9).min(1).values; ok_=(pos>=0)&(neg<9)
                    if ok_.any():
                        loss=F.relu(pos[ok_]-neg[ok_]+0.3).mean()+0.1*((W-torch.eye(D_,device=DEVICE))**2).mean()*D_
                        opt.zero_grad(); loss.backward(); opt.step()
            Wn=W.detach().cpu().numpy(); E2=l2n(FE@Wn).astype(np.float32)
            adapted_dv,adapted_te=eval_emb(E2,dv_m),eval_emb(E2,te_m)
            use_adapter=False; dlo_dv=dhi_dv=None
            if adapted_dv is not None:
                d_ap=adapted_dv['per_query_ap']-base_dv['per_query_ap']; _,dlo_dv,dhi_dv=boot_ci(d_ap,1000); use_adapter=bool(dlo_dv>0)
            E_use=E2 if use_adapter else FE
            final_te=adapted_te if use_adapter else base_te
            def pair_scores(E,mask):
                ix=np.where(mask)[0]
                if len(ix)<2: return np.asarray([]),np.asarray([],dtype=bool)
                S=E[ix]@E[ix].T; pid_,cam_=PDF.pid.values[ix],PDF.cam.values[ix]; iu=np.triu_indices(len(ix),1)
                same=(pid_[:,None]==pid_[None,:])[iu]; difcam=(cam_[:,None]!=cam_[None,:])[iu]; keep=difcam|~same
                return S[iu][keep],same[keep]
            s_dv,l_dv=pair_scores(E_use,dv_m); s_te,l_te=pair_scores(E_use,te_m)
            def f1_at(sc,lab,t):
                pred=sc>=t; tp=int((pred&lab).sum()); return 2*tp/max(int(pred.sum())+int(lab.sum()),1)
            grid=np.linspace(.50,.999,200); thr_p=float(max(grid,key=lambda t:f1_at(s_dv,l_dv,t))) if len(s_dv) else float(CONFIG8['person_match_cos'])
            f1_te=float(f1_at(s_te,l_te,thr_p)) if len(s_te) else 0.0; pred_te=s_te>=thr_p
            prec_te=float(((pred_te&l_te).sum())/max(pred_te.sum(),1)) if len(s_te) else 0.0
            PERSON={'source':src,'n_ids':n,'baseline':{'mAP':base_te['mAP'],'rank1':base_te['rank1']},'adapted':{'mAP':adapted_te['mAP'],'rank1':adapted_te['rank1']},'selected_model':'adapter' if use_adapter else 'frozen_backbone','use_adapter':bool(use_adapter),'validation_adapter_delta_mAP_ci':[dlo_dv,dhi_dv] if dlo_dv is not None else None,'calibrated_match_cos':thr_p,'test_pair_F1':f1_te,'test_pair_precision':prec_te,'note':'الاختبار النهائي لم يُستخدم لاختيار المهايئ أو العتبة؛ الأداء خاص بهذه البيانات والكاميرات.'}
            print(json.dumps(PERSON,ensure_ascii=False,indent=1)); save_json(RUN_DIR/'person_eval.json',PERSON)
            if use_adapter: torch.save({'W':torch.tensor(Wn),'run_id':RUN_ID,'eval':PERSON},MODELS_V8/'person_adapter.pt')
            status('ok',f"نموذج المشاة المختار: {PERSON['selected_model']} | عتبة={thr_p:.3f} | F1 اختبار={f1_te:.3f}")


In [ ]:
# ٨. تشغيل الفيديوهات: كشف + تتبع المشاة والمركبات ← هويات مجهولة ← مسارات في قاعدة بيانات (Drive) + CSV + ملخص
RUNS = []
if UNI is not None and CONFIG8['videos']:
    for v in CONFIG8['videos']:
        t0 = datetime.fromisoformat(v['start_utc']).replace(tzinfo=timezone.utc).timestamp() if v.get('start_utc') else None
        t_run = time.time(); out = UNI.process_video(v['path'], v['camera'], t0); out['seconds'] = time.time() - t_run; RUNS.append(out)
        print(f"{v['camera']}: مشاة={len(out['persons'])} مسار | مركبات={len(out['vehicles'])} مسار | {out['seconds']:.0f}s")
    purged = UNI.purge(); UNI.store.db.commit()
    atomic_copy(LOCAL / 'tracking.db', TRACK_DIR / 'tracking.db')
    n_p = UNI.store.export_csv(TRACK_DIR / 'person_trajectories.csv', 'person'); n_v = UNI.store.export_csv(TRACK_DIR / 'vehicle_trajectories.csv', 'vehicle')
    summ = UNI.store.summary(); save_json(RUN_DIR / 'tracking_summary.json', {'store': summ, 'runs': RUNS, 'purged_tracks': purged})
    print(json.dumps(summ, ensure_ascii=False)); status('ok', f'القاعدة: {TRACK_DIR / "tracking.db"} | نقاط مشاة={n_p} | نقاط مركبات={n_v} | حُذف بسبب الاحتفاظ: {purged}')
    if RUNS and RUNS[0]['persons']: print('مثال مشي:', RUNS[0]['persons'][0])
    if RUNS and RUNS[0]['vehicles']: print('مثال مركبة:', json.dumps(RUNS[0]['vehicles'][0], ensure_ascii=False)[:600])
    print('\nمسار هوية واحدة: UNI.store.trajectory("P-000001") | حذف هوية بطلب صاحبها: UNI.store.delete_entity("P-000001")')
else:
    status('pause', 'لا فيديوهات: أضفها في CONFIG8["videos"] = [{"path":..., "camera":"cam1", "start_utc":"2026-11-21T08:00:00"}]')


In [ ]:
# ٩. التقرير + الترقية الآمنة + خريطة ما في Drive
rows = []
def gate_row(name, res, ok, detail): rows.append({'الفحص': name, 'الحالة': 'PASS' if ok else ('NOT_EVALUATED' if not res else 'FAIL'), 'التفصيل': detail})
r = RETRAIN
gate_row('مركبات: تعلّم تراكمي بلا نسيان', r, bool(r and r.get('pass')),
         (f"v{r['version']} | طراز {r['fused_model_acc']:.3f} (CI {r['fused_ci'][0]:.3f}–{r['fused_ci'][1]:.3f}) | TTA={r['tta_flip']}" + (f" | نسيان {r['forgetting']['drop']:+.3f}" if r.get('forgetting') else '') + ('' if r['pass'] else ' | ' + ' | '.join(r['fail_reasons']))) if r else 'لا فئات مسمّاة كافية')
p = PERSON
gate_row('مشاة: مطابقة المظهر على بياناتك', p, bool(p and p.get('test_pair_F1', 0) > 0),
         (f"mAP {p['baseline']['mAP']:.3f}→{p['adapted']['mAP']:.3f} | عتبة معايرة {p['calibrated_match_cos']:.3f} | F1 {p['test_pair_F1']:.3f}") if p else 'لا بيانات مشاة مرخّصة — العتبة محافِظة وغير معايرة')
gate_row('تتبع فيديو', RUNS, bool(RUNS), f"{len(RUNS)} فيديو" if RUNS else 'لا فيديوهات')
report = pd.DataFrame(rows); print(report.to_string(index=False))
PROD_DIR.mkdir(parents=True, exist_ok=True); atomic_copy(DRIVE_ROOT / 'yemenvision_unified.py', PROD_DIR / 'yemenvision_unified.py')
if (MODELS_V8 / 'person_adapter.pt').is_file() and PERSON.get('selected_model') == 'adapter': atomic_copy(MODELS_V8 / 'person_adapter.pt', PROD_DIR / 'person_adapter.pt')
save_json(RUN_DIR / 'stage8_report.json', {'run_id': RUN_ID, 'retrain': RETRAIN, 'person': PERSON, 'mined': MINED, 'runs': [{k: v for k, v in x.items() if k not in ('persons', 'vehicles')} for x in RUNS]})
(RUN_DIR / 'stage8_report.md').write_text('# تقرير المرحلة ٨ — ' + RUN_ID + '\n\n' + report.to_string(index=False), encoding='utf-8')
print('\nما في Drive (كله دائم):')
for label, d in (('model_zoo/ — أوزان مفتوحة المصدر', ZOO), ('models_v8/ — نسخ البنك المُرقَّمة + مهايئ المشاة', MODELS_V8), ('production/ — الإنتاج', PROD_DIR), ('tracking/ — القاعدة والمسارات', TRACK_DIR), ('production_archive/ — القديم للتراجع', DRIVE_ROOT / 'production_archive')):
    n = sum(1 for p_ in d.rglob('*') if p_.is_file()) if d.exists() else 0; print(f'  {label}: {n} ملف')
status('ok', f'التقرير: {RUN_DIR / "stage8_report.md"}')


## بعد هذا الدفتر
1. شغّل الخلية 8 على إطارات من فيديواتك الحقيقية، سمِّ العناقيد الأكبر، ثم الخلية 9 — كرّر الدورة (كل دورة تضيف طرازات وتقيس النسيان).
2. اجمع صور مشاة **مرخّصة** (`<معرّف>/<كاميرا>_*.jpg`، ≥30 معرّفاً) لتقيس المطابقة وتعاير العتبة (الخلية 10).
3. ضع فيديو واحداً في `videos` وراجع `tracking/person_trajectories.csv` على مخطط (x,y) لتتحقق بالعين أن المسارات منطقية.
4. إن أردت مسارات بالأمتار: حدّد 4 نقاط معروفة الأبعاد على الأرض في `homography`.

## ضوابط الجودة والتشغيل — النسخة الاحترافية

- لا توجد طريقة تضمن نتائج ممتازة قبل إدخال بيانات حقيقية ممثلة للكاميرات والمركبات والمشاة. الدفتر يوفّر فحوصات وتوقفاً آمناً عند نقص المدخلات، لا دقة رقمية مسبقة.
- الافتراضي يستخدم YOLO11m COCO للمركبات والمشاة. لا تُحمّل أوزاناً مخصصة قديمة تلقائياً؛ فعّل use_custom_vehicle_detector فقط بعد التأكد من نوعها وتقييمها لفئات السيارة والدراجة والحافلة والشاحنة؛ لا يعادل كاشفاً مخصصاً لبيئة اليمن.
- مطابقة الأشخاص مجهولة فقط؛ لا تعرّف على الوجه ولا تربط الأرقام بأسماء. لا تخزّن صور المشاة. يلزم التفويض القانوني المناسب قبل تشغيل كاميرات حقيقية.
- لا تُرقِّ نموذجاً لمجرد أنه تدرب؛ يجب اجتياز بوابة الثقة وعدم التراجع عن النموذج السابق. عند نقص البيانات تظهر NOT_EVALUATED/FAIL بدلاً من ادعاء النجاح.
- ابدأ بـ 30–50 صورة متنوعة لكل فئة مركبة على الأقل، من أكثر من رحلة/كاميرا/إضاءة، مع اختبار مستقل زمنياً ومكانياً. للإنتاج الفعلي ستحتاج بيانات أكثر بكثير.
- لا تستخدم نتائج Market-1501 كدليل على صلاحية التشغيل في اليمن؛ الأفضل بيانات مرخّصة من بيئة الكاميرات المستهدفة.


## الإصلاحات في هذه النسخة
- نقل فحص ذاكرة OpenCLIP إلى ما قبل تحميل الوحدة، مع حذف ملف checkpoint المبتور وإعادة محاولة التحميل مرة واحدة عند فشل القراءة.
- إصلاح فحص إصدار NumPy؛ النسخة السابقة كانت تتجاوز شرط `<2.3` إذا كان NumPy موجوداً حتى لو كان أحدث من المطلوب.
- جعل YOLO11m هو الافتراضي للمركبات والمشاة، ومنع اختيار أوزان YOLO11s قديمة تلقائياً.
- إضافة خيار صريح لاستخدام كاشف مركبات مخصص بعد تحديد مساره يدوياً.
- جعل فشل كاشف/نموذج ReID/مهايئ اختياري لا يوقف بقية النظام متى أمكن، مع إظهار التحذير.
- إصلاح تعارض أبعاد بصمات قديمة في SQLite، وإزالة مقطع تقييم المشاة المكرر الذي كان يستدعي متغيرات غير معرّفة.
- مسح مخرجات التشغيل القديمة من الدفتر حتى لا تظهر أخطاء جلسة سابقة على أنها نتائج النسخة الجديدة.

**مهم:** هذه إصلاحات للكود ومسارات الفشل، وليست ضماناً لدقة أو نجاح تنزيل الأوزان في كل جلسة. يجب أن يتوفر اتصال تنزيل أول مرة، وملف بنك المرحلة 7 الصحيح، وذاكرة GPU كافية. بعد فتح النسخة، استخدم Runtime → Restart session ثم شغّل الخلايا من الأعلى بالترتيب.
